<a href="https://colab.research.google.com/github/prajeeta15/NVIDIA-Nemotron-Competition-Kaggle/blob/main/Nemotron_Submission_Prats.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

nvidia_nemotron_model_reasoning_challenge_path = kagglehub.competition_download('nvidia-nemotron-model-reasoning-challenge')
ryanholbrook_nvidia_utility_script_path = kagglehub.utility_script_install('ryanholbrook/nvidia-utility-script')
metric_nemotron_3_nano_30b_a3b_bf16_transformers_default_1_path = kagglehub.model_download('metric/nemotron-3-nano-30b-a3b-bf16/Transformers/default/1')

print('Data source import complete.')


In [ ]:
import polars as pl

train = pl.read_csv('/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv')

train.head()

**Reasoning**:
The first step is to inspect the `train` DataFrame to understand its structure and content, as per the instructions. This will involve displaying the first few rows and the schema.



In [ ]:
print("### Inspecting the train DataFrame\n")
print("#### First 5 rows:")
print(train.head())

print("\n#### DataFrame Schema:")
print(train.schema)

### Inspecting the train DataFrame

#### First 5 rows:


NameError: name 'train' is not defined

In [3]:
import site

cutlass_pkg_path = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/"
site.addsitedir(cutlass_pkg_path)

import kagglehub
# import mamba_ssm # Commented out as it's not found and not critical for this step
import torch
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer

# Configuration
# Use the model path from the initial download in T8wgs3jgfyCq
MODEL_PATH = metric_nemotron_3_nano_30b_a3b_bf16_transformers_default_1_path
OUTPUT_DIR = "/kaggle/working"
LORA_RANK = 32  # Can be set to a maximum of 32

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16
)
print("Model loaded successfully.")

# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

# Add a padding token if it's missing and resize model embeddings
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'}) # Or use a token already present if appropriate
    model.resize_token_embeddings(len(tokenizer))
print("Tokenizer loaded and configured successfully.")

# Initialize LoRA Adapter
print(f"Initializing LoRA adapter with rank={LORA_RANK}...")
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=16,
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


# YOUR CODE HERE
# --------------
# model.train()
# --------------


# Save Adapter
print(f"Saving adapter to {OUTPUT_DIR}...")
model.save_pretrained(OUTPUT_DIR)


NameError: name 'metric_nemotron_3_nano_30b_a3b_bf16_transformers_default_1_path' is not defined

In [ ]:
import subprocess

subprocess.run("zip -m submission.zip *", shell=True, check=True)

In [ ]:
print('Done.')

# Task
Fine-tune the Nemotron-3-Nano-30B model using LoRA with prompt engineering strategies (including Zero-shot, Few-shot, and Chain-of-Thought) to improve its reasoning capabilities, ensuring the model's output is formatted with `\boxed{answer}`, and establish a robust evaluation framework for comparing the fine-tuned model against a baseline on the `train.csv` dataset.

## Prepare Training Data for LoRA

### Subtask:
Analyze the loaded `train.csv` to understand its structure. Preprocess and format this data into prompt-response pairs suitable for LoRA fine-tuning, ensuring the desired output format `\boxed{answer}` is correctly integrated into the target labels. This will involve tokenization and potentially creating input_ids and attention_mask.


**Reasoning**:
The previous attempt to inspect the `train` DataFrame failed because the `train` variable was not defined in the execution environment. To resolve this, the `polars` library needs to be imported, and the `train` DataFrame reloaded within the same code block to ensure it's accessible before displaying its head and schema.



In [ ]:
import polars as pl

# Load the train DataFrame using the path corresponding to the competition slug
# used in kagglehub.competition_download in cell T8wgs3jgfyCq.
train = pl.read_csv('/kaggle/input/nvidia-nemotron-model-reasoning-challenge/train.csv')

print("### Inspecting the train DataFrame\n")
print("#### First 5 rows:")
print(train.head())

print("\n#### DataFrame Schema:")
print(train.schema)

FileNotFoundError: No such file or directory (os error 2): /kaggle/input/nvidia-nemotron-model-reasoning-challenge/train.csv

## Prepare Training Data for LoRA

### Subtask:
Analyze the loaded `train.csv` to understand its structure. Preprocess and format this data into prompt-response pairs suitable for LoRA fine-tuning, ensuring the desired output format `\boxed{answer}` is correctly integrated into the target labels. This will involve tokenization and potentially creating input_ids and attention_mask.


### Creating a Dummy `train` DataFrame for Development

In [5]:
# The content of this cell has been moved to cell 77293d92 to ensure the `train` DataFrame is defined before use.
# import polars as pl

# # Create a dummy DataFrame to simulate train.csv structure
# dummy_data = {
#     'id': [1, 2, 3, 4, 5],
#     'prompt': [
#         'What is 2 + 2?',
#         'Explain the concept of photosynthesis.',
#         'Who was Albert Einstein?',
#         'Calculate the area of a rectangle with length 5 and width 3.',
#         'What is the capital of France?'
#     ],
#     'completion': [
#         'The answer is 4.',
#         'Photosynthesis is the process by which green plants and some other organisms use sunlight to synthesize foods from carbon dioxide and water.',
#         'Albert Einstein was a German-born theoretical physicist who developed the theory of relativity, one of the two pillars of modern physics.',
#         'The area of the rectangle is 15.',
#         'The capital of France is Paris.'
#     ],
#     'answer': [
#         '\\boxed{4}',
#         '\\boxed{Photosynthesis is the process by which green plants and some other organisms use sunlight to synthesize foods from carbon dioxide and water.}',
#         '\\boxed{Albert Einstein}',
#         '\\boxed{15}',
#         '\\boxed{Paris}'
#     ]
# }

# train = pl.DataFrame(dummy_data)

# print("### Inspecting the dummy train DataFrame\n")
# print("#### First 5 rows:")
# print(train.head())

# print("\n#### DataFrame Schema:")
# print(train.schema)


### Preprocessing the Data for LoRA Fine-Tuning

In [4]:
import torch
import polars as pl
# MODEL_PATH and tokenizer are expected to be available from previous cells (e.g., RmNUyps3fyCs)

# Create a dummy DataFrame to simulate train.csv structure
dummy_data = {
    'id': [1, 2, 3, 4, 5],
    'prompt': [
        'What is 2 + 2?',
        'Explain the concept of photosynthesis.',
        'Who was Albert Einstein?',
        'Calculate the area of a rectangle with length 5 and width 3.',
        'What is the capital of France?'
    ],
    'completion': [
        'The answer is 4.',
        'Photosynthesis is the process by which green plants and some other organisms use sunlight to synthesize foods from carbon dioxide and water.',
        'Albert Einstein was a German-born theoretical physicist who developed the theory of relativity, one of the two pillars of modern physics.',
        'The area of the rectangle is 15.',
        'The capital of France is Paris.'
    ],
    'answer': [
        '\\boxed{4}',
        '\\boxed{Photosynthesis is the process by which green plants and some other organisms use sunlight to synthesize foods from carbon dioxide and water.}',
        '\\boxed{Albert Einstein}',
        '\\boxed{15}',
        '\\boxed{Paris}'
    ]
}

train = pl.DataFrame(dummy_data)

# Define a function to format the prompt-response pairs
def format_data_for_lora(sample):
    # The instruction can be a combination of prompt and completion
    # The target is the 'answer' column, formatted with \boxed{}

    # We need to create a template that the model will learn to follow
    # For fine-tuning, we concatenate the instruction and the target
    # and let the model learn to predict the target given the instruction.

    # Example template (this can be refined based on experimentation):
    instruction = f"### Instruction:\n{sample['prompt']}\n### Input:\n{sample['completion']}\n### Response:"
    target = f"{sample['answer']}{tokenizer.eos_token}" # Add EOS token to the target

    # Tokenize the instruction and target separately
    instruction_ids = tokenizer.encode(instruction, add_special_tokens=False)
    target_ids = tokenizer.encode(target, add_special_tokens=False)

    # Combine for training: Input will be instruction_ids + target_ids
    # Labels will be -100 for instruction_ids and target_ids for target_ids
    input_ids = instruction_ids + target_ids
    labels = [-100] * len(instruction_ids) + target_ids

    # Create attention mask
    attention_mask = [1] * len(input_ids)

    return {"input_ids": input_ids, "labels": labels, "attention_mask": attention_mask}

# Apply the formatting function to the dummy train DataFrame
# Polars DataFrames need to be converted to a list of dictionaries for easy iteration
# 'train' (dummy_train) is expected to be available from previous cells (e.g., 2446cf7f)
formatted_data = [format_data_for_lora(row) for row in train.iter_rows(has_header=True, return_type='dict')]

print("First formatted sample:\n", formatted_data[0])
print("Length of formatted data: ", len(formatted_data))

# Pad and create dataset (further steps will involve padding to a max length and converting to PyTorch Dataset)
# For now, we'll just show the raw token IDs


TypeError: DataFrame.iter_rows() got an unexpected keyword argument 'has_header'

# Task
**Task**: Load the tokenizer from the `MODEL_PATH` that was previously used to load the model. Add a padding token if missing, and resize the model's token embeddings to match the new vocabulary size. Then, prepare the `dummy_train` Polars DataFrame by tokenizing prompt-response pairs, creating `input_ids`, `labels`, and `attention_mask` for LoRA fine-tuning.

## Load Tokenizer and Ensure Model/Tokenizer Paths are Correct

### Subtask:
Load the tokenizer and ensure correct model paths, then preprocess and format the dummy train data for LoRA fine-tuning.
